In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Tutorial 04 — Optimization principles

SoftMobility wraps the soft-body rollout in a JAX-friendly objective so
that **design parameters** can be optimised by gradient descent.

This tutorial walks through the workflow on a small, fully-determined
example: an **asymmetric dumbbell** sinking under gravity. Two spheres
are placed on either side of the origin. The radius of the left sphere
is fixed at `a_0 = 1`; we optimize the radius `a_1` of the right one so
that the dumbbell falls **vertically** (zero lateral drift) instead of
tipping under unequal drag.

We use:

* `softmobility.SoftBody` for the body,
* `softmobility.FlowBodyRollout` for the trajectory,
* `softmobility.FlowBodyOptimizer` to wrap an Optax optimiser around a
  scalar objective.


In [11]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go

import softmobility as sm

np.set_printoptions(precision=3, suppress=True)

## Step 1 — Body

The right-sphere radius `a_1` is the only design parameter. Each sphere
carries a gravity-coupled body force `m·g` with `m = (4/3)π a³`:

In [12]:
yaml_body = """
design_names: [a_1]
input_names: [gravity]
defaults:
  a_1: 0.5
  K: 4.18879  # 4*pi/3

spheres:
  - radius: 1.0  # a_0
    position: [-1, 0, 0]
    force: [K * gravity0, K * gravity1, K * gravity2]
  - radius: a_1  # a_1
    position: [ 1, 0, 0]
    force: [K * a_1**3 * gravity0, K * a_1**3 * gravity1, K * a_1**3 * gravity2]
"""

body = sm.SoftBody(yaml_body, verbose=False)
print(repr(body))

SPHERE ASSEMBLY
  2 spheres
  0 degrees of freedom
  1 design parameters
  3 input parameters

Default values
  degrees of freedom dof: [] = []
  design parameters param: ['a_1'] = [0.5]
  input parameters param: ['gravity0', 'gravity1', 'gravity2']

SPHERE 0
  radius: 1.0
  position: [-1.  0.  0.]
  orientation: [0. 0. 0.]
  C_H:
[[4.189 0.    0.   ]
 [0.    4.189 0.   ]
 [0.    0.    4.189]
 [0.    0.    0.   ]
 [0.    0.    0.   ]
 [0.    0.    0.   ]]
  C_K:
[]

SPHERE 1
  radius: 0.5
  position: [1. 0. 0.]
  orientation: [0. 0. 0.]
  C_H:
[[0.524 0.    0.   ]
 [0.    0.524 0.   ]
 [0.    0.    0.524]
 [0.    0.    0.   ]
 [0.    0.    0.   ]
 [0.    0.    0.   ]]
  C_K:
[]



## Step 2 — Rollout

Gravity along $-\boldsymbol{e}_3$, no flow. The body starts horizontal (along $\boldsymbol{e}_1$). With unequal radii the heavier sphere experiences more
gravitational force but its drag scales differently (mass ∝ r³, drag ∝
r), so the body tilts and drifts laterally.


In [13]:
rollout = sm.FlowBodyRollout(
    soft_body=body,
    flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=1)},
)

dt = 0.005
n_steps = 400

## Step 3 — Define the scalar objective

We want the dumbbell to fall vertically: minimise the **lateral drift**
of the body-reference point at the end of the rollout.

The objective signature must be `(rollout, design) -> scalar` so JAX can
trace and differentiate it.


In [14]:
def lateral_drift_squared(rollout, design):
    positions, _, _ = rollout.rollout(
        dt=dt,
        n_steps=n_steps,
        design=design,
    )
    final = positions[-1]
    return final[0] ** 2 + final[1] ** 2  # ignore the e_3 (vertical) component


# Sanity check: gradient at the symmetric point a_1 = 1 should be ~0.
g0 = jax.grad(lambda d: lateral_drift_squared(rollout, d))(jnp.array([0.5]))
g1 = jax.grad(lambda d: lateral_drift_squared(rollout, d))(jnp.array([1.0]))
print(f"∂/∂a_1 of drift² at a_1=0.5 : {float(g0[0]):.4e}")
print(f"∂/∂a_1 of drift² at a_1=1.0 : {float(g1[0]):.4e}")

∂/∂a_1 of drift² at a_1=0.5 : -5.8328e-06
∂/∂a_1 of drift² at a_1=1.0 : 0.0000e+00


## Step 4 — Run the optimiser

Adam, learning rate 0.05, 60 steps. `a_1` is clipped to `[0.1, 10]`.


In [17]:
optimizer = sm.FlowBodyOptimizer(
    rollout,
    lateral_drift_squared,
    optax.adam(learning_rate=0.1),
)

init_design = jnp.array([0.5])
optimal_design = optimizer.run(
    init_design=init_design,
    n_steps=100,
    print_every=10,
    clip_min=0.1,
    clip_max=10.0,
    maximize=False,
)
print(f"\nfinal a_1 = {float(optimal_design[0]):.4f}  (expected ≈ 1.0 for symmetric dumbbell)")


step    0  objective=-0.0000  |grad|=0.0000  design0=0.60
step   10  objective=-0.0000  |grad|=0.0000  design0=1.17
step   20  objective=-0.0000  |grad|=0.0000  design0=0.90
step   30  objective=-0.0000  |grad|=0.0000  design0=1.06
step   40  objective=-0.0000  |grad|=0.0000  design0=0.97
step   50  objective=-0.0000  |grad|=0.0000  design0=1.01
step   60  objective=-0.0000  |grad|=0.0000  design0=1.00
step   70  objective=-0.0000  |grad|=0.0000  design0=0.99
step   80  objective=-0.0000  |grad|=0.0000  design0=1.00
step   90  objective=-0.0000  |grad|=0.0000  design0=1.00
step   99  objective=-0.0000  |grad|=0.0000  design0=1.00

final a_1 = 0.9991  (expected ≈ 1.0 for symmetric dumbbell)


## Step 5 — Compare trajectories

Plot the centre-of-mass trajectory before vs after optimisation. The
initial asymmetric body drifts sideways while the optimised body falls
vertically.


In [21]:
def trajectory(design):
    positions, _, _ = rollout.rollout(
        dt=dt,
        n_steps=n_steps,
        design=design,
    )
    return np.asarray(positions)


traj_init = trajectory(init_design)
traj_opt = trajectory(optimal_design)

fig = go.Figure()
fig.add_trace(go.Scatter(x=traj_init[:, 0], y=traj_init[:, 2],
                         mode="lines",
                         name=f"initial r₁ = {float(init_design[0]):.2f}",
                         line=dict(color="#d62728", width=3)))
fig.add_trace(go.Scatter(x=traj_opt[:, 0], y=traj_opt[:, 2],
                         mode="lines",
                         name=f"optimised r₁ = {float(optimal_design[0]):.3f}",
                         line=dict(color="#1f77b4", width=3)))
fig.update_layout(
    title="Centre-of-mass trajectory before / after optimisation",
    xaxis_title="x  (lateral drift)",
    yaxis_title="z  (vertical fall)",
    yaxis=dict(scaleanchor="x", scaleratio=0.01),
    width=700, height=500,
    plot_bgcolor="white",
)
fig.show()


## Notes

* Here the body is **rigid** (no DOFs); only the design `a_1` changes.
  The same workflow applies when the body has DOFs: the rollout
  integrates them, the optimizer only sees the scalar objective.
* The optimizer uses `jax.value_and_grad`, so the objective must be
  built **purely** from JAX operations on the rollout output.